<a href="https://colab.research.google.com/github/SlLeonn/SenalesySistemas/blob/main/TallerSLIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# ***Punto #1***
---

### Evaluación de la convolución $ y(t) = x(t) * h(t) $

Dadas las señales:

- $ x(t) = \text{rect}\left(t - \frac{1}{2}\right) = u(t) - u(t - 1) $
- $ h(t) = e^{-t} u(t) $

La convolución se define como:

$$
y(t) = \int_{-\infty}^{\infty} x(\tau) h(t - \tau) \, d\tau
$$

---

#### **Caso 1: $ t < 0 $**

No hay traslape entre $ x(\tau) $ y $ h(t - \tau) $, entonces:

$$
y(t) = 0
$$

---

#### **Caso 2: $ 0 \leq t < 1 $**

El intervalo de integración es desde \( \tau = 0 \) hasta \( \tau = t \):

$$
y(t) = \int_0^t e^{-(t - \tau)} \, d\tau = e^{-t} \int_0^t e^{\tau} \, d\tau = e^{-t} (e^t - 1) = 1 - e^{-t}
$$

---

#### **Caso 3: $ t \geq 1 $**

Ahora el traslape ocurre completamente en \( \tau \in [0, 1] \):

$$
y(t) = \int_0^1 e^{-(t - \tau)} \, d\tau = e^{-t} \int_0^1 e^{\tau} \, d\tau = e^{-t}(e - 1)
$$

---

### Resultado final:

$$
y(t) =
\begin{cases}
0, & t < 0 \\\\
1 - e^{-t}, & 0 \leq t < 1 \\\\
(e - 1)e^{-t}, & t \geq 1
\end{cases}
$$

---
# ***Punto #2***
---

---
- Compare la señal de salida obtenida al resolver la EDO con la obtenida mediante la convolución. Son iguales?
---

In [ ]:
import sympy as sym
import numpy as np
import matplotlib.pyplot as plt
from sympy import Heaviside, DiracDelta

# Configurar impresión bonita
sym.init_printing()

# Variables simbólicas
t, tau = sym.symbols('t tau', real=True)
x = sym.exp(-2*t) * sym.Heaviside(t)

# EDO: y' + y = x(t)
y = sym.Function('y')(t)
ode = sym.Eq(y.diff(t) + y, x)
sol_edo = sym.dsolve(ode)
C1 = sym.symbols('C1')
C1_val = sym.solve(sol_edo.rhs.subs(t, 0), C1)[0]
y_edo = sol_edo.rhs.subs(C1, C1_val)

# Mostrar solución simbólica de la EDO
display(sym.Eq(sym.Function('y')(t), y_edo.simplify()))

# Respuesta al impulso
h = sym.Function('h')(t)
impulse_ode = sym.Eq(h.diff(t) + h, DiracDelta(t))
sol_h = sym.dsolve(impulse_ode)
C2 = sym.symbols('C1')
C2_val = sym.solve(sol_h.rhs.limit(t, 0, '-'), C2)[0]
h_t = sol_h.rhs.subs(C2, C2_val)

# Mostrar respuesta al impulso h(t)
display(sym.Eq(sym.Function('h')(t), h_t.simplify()))

# Convolución: y(t) = ∫₀ᵗ x(τ) h(t - τ) dτ
x_tau = sym.exp(-2*tau)
h_shifted = h_t.subs(t, t - tau).subs(Heaviside(t), 1)
y_conv = sym.integrate(x_tau * h_shifted, (tau, 0, t))

# Mostrar resultado de la convolución simbólicamente
display(sym.Eq(sym.Function('y')(t), y_conv.simplify()))

# Funciones numéricas
y_edo_func = sym.lambdify(t, y_edo, 'numpy')
y_conv_func = sym.lambdify(t, y_conv, 'numpy')

# Tiempo y gráficos
t_vals = np.linspace(0, 10, 400)

plt.figure(figsize=(8, 4))
plt.plot(t_vals, y_edo_func(t_vals), label='y(t) por EDO', color='blue')
plt.plot(t_vals, y_conv_func(t_vals), label='y(t) por Convolución', color='red', linestyle='--')
plt.xlabel('t')
plt.ylabel('y(t)')
plt.title('Comparación de la salida: EDO vs Convolución')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

---
- Compruebe la solución $ h(t) $  de la EDO cuando $ x(t)=δ(t) $ de manera manual. Tener en cuenta que  $ (d/dt) $ $ ϵ(t)=δ(t) $.
---

### Verificación manual de la solución $ h(t) = e^{-t} u(t) $ para la EDO

Queremos comprobar que la función

$$
h(t) = e^{-t} u(t)
$$

satisface la ecuación diferencial:

$$
\frac{d}{dt} h(t) + h(t) = \delta(t)
$$

donde $ u(t) $ es la función escalón (Heaviside), y se cumple:

$$
\frac{d}{dt} u(t) = \delta(t)
$$

---

#### Paso 1: Derivada de $ h(t) $

Aplicando la regla del producto:

$$
\frac{d}{dt} h(t) = \frac{d}{dt} \left( e^{-t} u(t) \right)
= \left( \frac{d}{dt} e^{-t} \right) u(t) + e^{-t} \frac{d}{dt} u(t)
= -e^{-t} u(t) + e^{-t} \delta(t)
$$

---

#### Paso 2: Sustituimos en la EDO

Sumamos $ \frac{d}{dt} h(t) + h(t) $:

$$
\left( -e^{-t} u(t) + e^{-t} \delta(t) \right) + e^{-t} u(t)
= e^{-t} \delta(t)
$$

---

#### Paso 3: Evaluar en \( t = 0 \)

Como $ e^{-t} \delta(t) = \delta(t) $, porque la delta "extrae" el valor en $ t = 0 $:

$$
e^{-t} \delta(t) = e^0 \cdot \delta(t) = \delta(t)
$$

---

### ✅ Conclusión

Hemos comprobado que:

$$
\frac{d}{dt} h(t) + h(t) = \delta(t)
$$

por lo tanto, $ h(t) = e^{-t} u(t) $ **es solución** de la EDO con entrada impulso.

---
- Comprobar la solución de la integral de convolución de manera manual. Tener en cuenta las funciones Heaviside
---

### Verificación manual de la solución de la convolución

Queremos comprobar que la convolución:

$$
y(t) = x(t) * h(t) = \int_0^t x(\tau) \cdot h(t - \tau) \, d\tau
$$

con las señales:

- Entrada:
  $$
  x(t) = e^{-2t} u(t)
  $$

- Respuesta al impulso:
  $$
  h(t) = e^{-t} u(t)
  $$

produce:

$$
y(t) = (e^{-t} - e^{-2t}) u(t)
$$

---

#### Sustitución en la integral de convolución

$$
y(t) = \int_0^t e^{-2\tau} \cdot e^{-(t - \tau)} u(t - \tau) \, d\tau
$$

Sabemos que $ u(t - \tau) = 1 $ en el intervalo $ 0 \leq \tau \leq t $, entonces:

$$
y(t) = \int_0^t e^{-2\tau} \cdot e^{-(t - \tau)} \, d\tau = e^{-t} \int_0^t e^{-\tau} \, d\tau
$$

Calculamos la integral:

$$
\int_0^t e^{-\tau} \, d\tau = \left[ -e^{-\tau} \right]_0^t = -e^{-t} + 1
$$

Por lo tanto:

$$
y(t) = e^{-t} (1 - e^{-t}) = e^{-t} - e^{-2t}
$$

---

### ✅ Conclusión

Hemos comprobado manualmente que:

$$
x(t) * h(t) = (e^{-t} - e^{-2t}) u(t)
$$

lo cual coincide con la solución obtenida resolviendo la EDO.

---
# ***Punto #3***
---

### 1. Sea el sistema definido por la ecuación en diferencias:

$$
y[n] = \frac{1}{3}x[n] + 2x[n-1] - y[n-1]
$$

Queremos analizar si el sistema es **lineal** e **invariante en el tiempo** (SLIT).


In [ ]:
import sympy as sp

# Variables simbólicas
n, n0 = sp.symbols('n n0', integer=True)
a, b = sp.symbols('a b')  # escalares

# Funciones de entrada y salida
x = sp.Function('x')
y = sp.Function('y')
x1 = sp.Function('x1')
x2 = sp.Function('x2')
y1 = sp.Function('y1')
y2 = sp.Function('y2')

# ----------------------------
# 📘 Ecuación original del sistema
# y[n] = (1/3)x[n] + 2x[n-1] - y[n-1]
# ----------------------------
sistema = sp.Eq(y(n), (1/3)*x(n) + 2*x(n - 1) - y(n - 1))
print("📘 Ecuación del sistema:")
sp.pprint(sistema, use_unicode=True)

# ----------------------------
# 🔍 Verificación de Linealidad
# ----------------------------

# Supuestas salidas individuales
y1_expr = (1/3)*x1(n) + 2*x1(n - 1) - y1(n - 1)
y2_expr = (1/3)*x2(n) + 2*x2(n - 1) - y2(n - 1)

# Entrada combinada
x_total = a*x1(n) + b*x2(n)
y_total = (1/3)*x_total + 2*(a*x1(n - 1) + b*x2(n - 1)) - (a*y1(n - 1) + b*y2(n - 1))

# Combinación esperada de salidas
y_esperada = a*y1_expr + b*y2_expr

# Comparación
linealidad = sp.simplify(y_total - y_esperada) == 0
print("\n🔍 ¿Es el sistema lineal?:", linealidad)

# ----------------------------
# 🕒 Verificación de Invarianza en el tiempo
# ----------------------------

# Salida a entrada desplazada
y_shifted_input = (1/3)*x(n - n0) + 2*x(n - 1 - n0) - y(n - 1 - n0)

# Salida original desplazada
y_original = (1/3)*x(n) + 2*x(n - 1) - y(n - 1)
y_shifted_expected = y_original.subs(n, n - n0)

# Comparación
invarianza = sp.simplify(y_shifted_input - y_shifted_expected) == 0
print("🕒 ¿Es el sistema invariante en el tiempo?:", invarianza)

### 2. Sea el sistema:
$$
y[n] = \sum_{k=-\infty}^{n} x^2[k]
$$
Queremos analizar si el sistema es lineal e invariante en el tiempo (SLIT).

In [ ]:
# Variables simbólicas
n, k, l, n0 = sp.symbols('n k l n0', integer=True)
a, b = sp.symbols('a b')
x = sp.Function('x')
x1 = sp.Function('x1')
x2 = sp.Function('x2')

# ----------------------------
# Ecuación del sistema
# ----------------------------
y = sp.Function('y')
suma_general = sp.Sum(x(k)**2, (k, -sp.oo, n))
print("📘 Ecuación del sistema:")
sp.pprint(sp.Eq(y(n), suma_general), use_unicode=True)

# ----------------------------
# Verificación de LINEALIDAD
# ----------------------------
x_total = a * x1(k) + b * x2(k)
y_total = sp.summation((x_total)**2, (k, -sp.oo, n))
y_esperada = a * sp.summation(x1(k)**2, (k, -sp.oo, n)) + b * sp.summation(x2(k)**2, (k, -sp.oo, n))

es_lineal = sp.simplify(y_total - y_esperada) == 0
print("\n🔍 ¿Es el sistema lineal?:", es_lineal)

# ----------------------------
# Verificación manual de INVARIANZA
# ----------------------------

# Entrada desplazada: x[n - n0]
# Salida: sum_{k=-∞}^{n} x^2[k - n0]
# Hacemos cambio de variable: l = k - n0 ⟹ k = l + n0
# Resultado: sum_{l=-∞}^{n - n0} x^2[l]

lhs = sp.Sum(x(l)**2, (l, -sp.oo, n - n0))
rhs = sp.Sum(x(k - n0)**2, (k, -sp.oo, n)).rewrite(sp.Sum, l + n0)

print("\n🕒 Verificación manual:")
print("Forma esperada con cambio de variable:")
sp.pprint(lhs, use_unicode=True)

print("Forma obtenida desde entrada desplazada:")
sp.pprint(rhs, use_unicode=True)

print("\n✅ Conclusión: Las dos expresiones son matemáticamente equivalentes, aunque SymPy no devuelve True automáticamente.")

### 3. Dado el sistema:

$$
y[n] = \text{median}(x[n-1], x[n], x[n+1])
$$

donde `median` representa la mediana de una **ventana deslizante de tamaño 3**.

Queremos verificar si es un sistema **SLIT** (lineal e invariante en el tiempo).

In [ ]:
# Variables simbólicas
n, n0 = sp.symbols('n n0', integer=True)
a, b = sp.symbols('a b')
x = sp.Function('x')
x1 = sp.Function('x1')
x2 = sp.Function('x2')

# ----------------------------
# Definimos la "mediana simbólica" como una función formal
# ----------------------------
median3 = sp.Function('median3')

# Sistema: y[n] = median(x[n-1], x[n], x[n+1])
y = sp.Function('y')
y_n = y(n)
sistema = sp.Eq(y_n, median3(x(n - 1), x(n), x(n + 1)))

print("📘 Ecuación del sistema:")
sp.pprint(sistema, use_unicode=True)

# ----------------------------
# Verificación de LINEALIDAD simbólica (representación formal)
# ----------------------------
# Entrada combinada
x_comb = lambda k: a * x1(k) + b * x2(k)
y_comb = median3(x_comb(n - 1), x_comb(n), x_comb(n + 1))

# Combinación de salidas individuales (incorrecta para median, pero útil para probar linealidad)
y1 = median3(x1(n - 1), x1(n), x1(n + 1))
y2 = median3(x2(n - 1), x2(n), x2(n + 1))
y_esperada = a * y1 + b * y2

# Comparación (no se espera que sea igual)
es_lineal = sp.simplify(y_comb - y_esperada) == 0
print("\n🔍 ¿Es el sistema lineal?:", es_lineal)

# ----------------------------
# Verificación de INVARIANZA EN EL TIEMPO
# ----------------------------
x_shifted = lambda k: x(k - n0)
y_shifted = median3(x_shifted(n - 1), x_shifted(n), x_shifted(n + 1))
y_expected = median3(x(n - 1 - n0), x(n - n0), x(n + 1 - n0))

es_invariante = sp.simplify(y_shifted - y_expected) == 0
print("🕒 ¿Es el sistema invariante en el tiempo?:", es_invariante)

### 4. Dado el sistema:

$$
y(t) = A x(t) + B \quad ; \quad A, B \in \mathbb{R}
$$

Queremos analizar si es un sistema **SLIT** (lineal e invariante en el tiempo).

In [ ]:
# Variables simbólicas
t, t0 = sp.symbols('t t0', real=True)
A, B, a, b = sp.symbols('A B a b', real=True)
x = sp.Function('x')
x1 = sp.Function('x1')
x2 = sp.Function('x2')
y = sp.Function('y')

# ----------------------------
# Definición del sistema
# ----------------------------
y_t = A * x(t) + B
print("📘 Ecuación del sistema:")
sp.pprint(sp.Eq(y(t), y_t), use_unicode=True)

# ----------------------------
# 🔍 Verificación de LINEALIDAD
# ----------------------------
# Entrada combinada
x_total = a * x1(t) + b * x2(t)
y_total = A * x_total + B

# Salidas individuales combinadas
y1 = A * x1(t) + B
y2 = A * x2(t) + B
y_esperada = a * y1 + b * y2

es_lineal = sp.simplify(y_total - y_esperada) == 0
print("\n🔍 ¿Es el sistema lineal?:", es_lineal)

# ----------------------------
# 🕒 Verificación de INVARIANZA
# ----------------------------
# Entrada desplazada
x_shifted = x(t - t0)
y_shifted = A * x_shifted + B
y_expected = y_t.subs(t, t - t0)

es_invariante = sp.simplify(y_shifted - y_expected) == 0
print("🕒 ¿Es el sistema invariante en el tiempo?:", es_invariante)

---
# ***Punto #4***
---

## 📌 Datos

**Entrada:**

$$
x[n] = \{-15,\, 5,\, -3,\, 0,\, 5,\, 7,\, -1\}, \quad x[0] = -3
$$

**Respuesta al impulso:**

$$
h[n] = \{1,\, -2,\, 0,\, 1,\, -2\}, \quad h[0] = 0
$$

---

## 🔢 1. Reorganización con índices

Ya que se da que $x[0] = -3$, los índices de $x[n]$ son:

$$
\begin{array}{c|ccccccc}
n & -3 & -2 & -1 & 0 & 1 & 2 & 3 \\
x[n] & -15 & 5 & -3 & 0 & 5 & 7 & -1 \\
\end{array}
$$

Y como $h[0] = 0$, los índices de $h[n]$ son:

$$
\begin{array}{c|ccccc}
n & -2 & -1 & 0 & 1 & 2 \\
h[n] & 1 & -2 & 0 & 1 & -2 \\
\end{array}
$$

---

## 🔁 2. Convolución discreta: $y[n] = x[n] * h[n]$

La convolución discreta para un sistema SLIT se define como:

$$
y[n] = \sum_{k=-\infty}^{\infty} x[k] \cdot h[n - k]
$$

Como $x[n]$ y $h[n]$ son finitos, solo se suman los productos donde ambos están definidos.

---

## 📉 3. Derivación de $h[n]$ desde la respuesta al escalón $s[n]$

Si se conoce la **respuesta al escalón** $s[n]$, se puede obtener la respuesta al impulso mediante la relación:

$$
h[n] = s[n] - s[n - 1]
$$

Asumiendo que $s[-1] = 0$ como condición inicial.


In [ ]:
# Señal de entrada x[n]
x = np.array([-15, 5, -3, 0, 5, 7, -1])
n_x = np.arange(-3, 4)

# Respuesta al impulso h[n]
h = np.array([1, -2, 0, 1, -2])
n_h = np.arange(-2, 3)

# Convolución y[n] = x[n] * h[n]
y = np.convolve(x, h)
n_y = np.arange(n_x[0] + n_h[0], n_x[-1] + n_h[-1] + 1)

# Mostrar resultado
print("🔁 Salida y[n] = x[n] * h[n]:")
print(f"Índices: {n_y.tolist()}")
print(f"y[n]    = {y.tolist()}")

# Graficar y[n]
plt.figure(figsize=(10, 4))
plt.stem(n_y, y)
plt.title('Salida del sistema con respuesta al impulso')
plt.xlabel('n')
plt.ylabel('y[n]')
plt.grid(True)
plt.show()

# Respuesta al escalón dada
s = np.array([-1, 6, -10, 3, 1, -10, 2, 5])
s_prev = np.concatenate(([0], s[:-1]))
h_from_step = s - s_prev
n_hs = np.arange(0, len(h_from_step))

# Nueva convolución con h[n] derivada del escalón
y2 = np.convolve(x, h_from_step)
n_y2 = np.arange(n_x[0] + n_hs[0], n_x[-1] + n_hs[-1] + 1)

# Mostrar resultado
print("\n📉 Salida y2[n] con h[n] derivada del escalón:")
print(f"Índices: {n_y2.tolist()}")
print(f"y2[n]   = {y2.tolist()}")

# Graficar y2[n]
plt.figure(figsize=(10, 4))
plt.stem(n_y2, y2)
plt.title('Salida del sistema con h[n] derivada del escalón')
plt.xlabel('n')
plt.ylabel('y[n]')
plt.grid(True)
plt.show()


---
# ***Punto #5***
---

## 🔍 Análisis detallado de sistemas en serie con señal Gaussiana

---

### 📌 Datos del problema

- Entrada:  
  $$
  x(t) = e^{-a t^2}, \quad a \in \mathbb{R}^+
  $$

- Sistema $A$:  
  $$
  y_A(t) = x^2(t)
  $$

- Sistema $B$: SLIT con respuesta al impulso:  
  $$
  h_B(t) = B e^{-b t^2}, \quad b \in \mathbb{R}^+
  $$

Queremos analizar dos configuraciones de sistemas en serie:

---

## 🔷 a) Orden: $x(t) \rightarrow B \rightarrow A \rightarrow y(t)$

### Paso 1: Convolución $x(t) * h_B(t)$

Tenemos:

$$
x(t) = e^{-a t^2}, \quad h_B(t) = B e^{-b t^2}
$$

Entonces:

$$
z(t) = x(t) * h_B(t) = \int_{-\infty}^{\infty} e^{-a \tau^2} \cdot B e^{-b (t - \tau)^2} \, d\tau
$$

Esto es una **convolución de dos gaussianas**. El resultado conocido es:

$$
e^{-a t^2} * e^{-b t^2} = \sqrt{\frac{\pi}{a + b}} \cdot e^{-\frac{ab}{a + b} t^2}
$$

Aplicando esto:

$$
z(t) = B \sqrt{\frac{\pi}{a + b}} \cdot e^{-\frac{ab}{a + b} t^2}
$$

---

### Paso 2: Aplicar el sistema $A$: elevar al cuadrado

$$
y(t) = z^2(t) = \left( B \sqrt{\frac{\pi}{a + b}} \cdot e^{-\frac{ab}{a + b} t^2} \right)^2
$$

Entonces:

$$
y(t) = \frac{\pi B^2}{a + b} \cdot e^{-\frac{2ab}{a + b} t^2}
$$

---

## 🔷 b) Orden: $x(t) \rightarrow A \rightarrow B \rightarrow y(t)$

### Paso 1: Aplicar el sistema no lineal $A$

$$
y_A(t) = x^2(t) = \left( e^{-a t^2} \right)^2 = e^{-2a t^2}
$$

### Paso 2: Convolución con $h_B(t)$

Ahora:

$$
y(t) = y_A(t) * h_B(t) = \int_{-\infty}^{\infty} e^{-2a \tau^2} \cdot B e^{-b (t - \tau)^2} \, d\tau
$$

Esta también es una convolución de gaussianas. Aplicando la fórmula:

$$
e^{-2a t^2} * e^{-b t^2} = \sqrt{\frac{\pi}{2a + b}} \cdot e^{- \frac{2ab}{2a + b} t^2}
$$

Por lo tanto:

$$
y(t) = B \sqrt{ \frac{\pi}{2a + b} } \cdot e^{- \frac{2ab}{2a + b} t^2}
$$

---

## ✅ Conclusión final

| Configuración | Resultado de $y(t)$ |
|---------------|---------------------|
| $x \rightarrow B \rightarrow A$ | $\displaystyle y(t) = \frac{\pi B^2}{a + b} \cdot e^{- \frac{2ab}{a + b} t^2}$ |
| $x \rightarrow A \rightarrow B$ | $\displaystyle y(t) = B \sqrt{ \frac{\pi}{2a + b} } \cdot e^{- \frac{2ab}{2a + b} t^2}$ |

---

### ❗ Observación

- El orden de los sistemas afecta la salida porque **$A$ no es lineal**, y la propiedad conmutativa de la convolución **no se cumple** si hay un sistema no lineal involucrado.
- En consecuencia, **los sistemas en serie no conmutan** si uno de ellos es no lineal.

---